# Model V1

In [ ]:
import sys
sys.path.append('../data_extraction')

import os
import numpy as np
import mlflow
import keras
import tensorflow as tf

import mlflow.keras
from keras.layers import Conv2D, MaxPooling2D, Conv2DTranspose, Concatenate, Dropout
from keras.regularizers import l2

from LRO_data_class import getSplitIndices, getNormalisedBatch, getAugmentedBatch, patchGenerator, stepsPerEpoch

print(tf.config.list_physical_devices('GPU'))

/Users/none/miniconda3/envs/lunar_lro/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-08-14 08:48:05.782074: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
train_idx, val_idx, test_idx = getSplitIndices()
print(f'train: {len(train_idx)}  val: {len(val_idx)}  test: {len(test_idx)}')

wac_train, dem_train, mask_train = getAugmentedBatch(batch_num=0)

# validation/test: normalised only, no augmentation
wac_val, dem_val, mask_val = getNormalisedBatch(batch_num=0)

# wac  (1000, 256, 256) float32 [0, 1]  - per-patch percentile normalised
# dem  (1000, 256, 256) float32 [0, 1]  - per-patch percentile normalised
# mask (1000, 256, 256) uint8   {0, 1}  - 1 px ring masks, ~37:1 imbalance
print(wac_train.shape, wac_train.dtype, wac_train.min(), wac_train.max())

## Hyperparameters

In [ ]:
# all hyperparameters in one place — change here and MLflow logs them automatically

# [LOSS - still undecided, see global_project_notes.md 12]
    # Masks are 37:1 background:crater (2.6% crater pixels, measured 2026-07-20).
    # The 2026-08-11 run plateaued at ~0.134, which is close to the BCE of simply
    # predicting the 2.6% base rate everywhere. That looked like imbalance collapse,
    # BUT the run was only 50 gradient updates with augmentation left on, so it does
    # not distinguish "stuck" from "undertrained" - conclusion withdrawn 2026-08-14.
    # DeepMoon used plain UNWEIGHTED BCE (paper 2.6 eq.7 + model_train.py), so any
    # weighting/focal here is a deliberate divergence, not replication.
    # Focal is wired up below because it is built into Keras - not on evidence.

params = {
    'dim': 256,
    'channels': 'both',                 # 'both' | 'wac' | 'dem'
    'input_channels': 2,                # 2 for both, 1 for ablations
    'n_filters': 32,                    # DeepMoon used 112 (paper 2.3)
    'FL': 3,                            # kernel size
    'init': 'he_normal',
    'lmbda': 1e-6,                      # L2. NB paper 2.7 says 1e-5, repo says 1e-6 - sources disagree
    'dropout': 0.15,
    'learning_rate': 0.0001,
    'batch_size': 8,
    'epochs': 50,
    'loss': 'binary_focal_crossentropy',
    'focal_alpha': 0.25,
    'focal_gamma': 2.0,
    'focal_class_balancing': True,
    'model': 'U-Net-v1',
}

## Model Architecture

In [ ]:
img_input = keras.Input(shape=(params['dim'], params['dim'], params['input_channels']))

# Encoder1
a1 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(img_input)

a1 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a1)

a1P = MaxPooling2D((2, 2), strides=(2, 2))(a1)

# Encoder2
a2 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a1P)

a2 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a2)

a2P = MaxPooling2D((2, 2), strides=(2, 2))(a2)

# Encoder3
a3 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a2P)

a3 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a3)

a3P = MaxPooling2D((2, 2), strides=(2, 2))(a3)

# Encoder4
a4 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a3P)

a4 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a4)

a4P = MaxPooling2D((2, 2), strides=(2, 2))(a4)

u = Conv2D(
    params['n_filters'] * 16, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(a4P)

u = Conv2D(
    params['n_filters'] * 16, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(u)

# Decoder1
d1CT = Conv2DTranspose(params['n_filters']*8, kernel_size=2, strides=2, padding='same')(u)
d1c = Concatenate()([d1CT, a4])
x1 = Dropout(params['dropout'])(d1c)

d1 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x1)

d1 = Conv2D(
    params['n_filters'] * 8, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d1)

# Decoder2
d2CT = Conv2DTranspose(params['n_filters']*4, kernel_size=2, strides=2, padding='same')(d1)
d2c = Concatenate()([d2CT, a3])
x2 = Dropout(params['dropout'])(d2c)

d2 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x2)

d2 = Conv2D(
    params['n_filters'] * 4, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d2)

# Decoder3
d3CT = Conv2DTranspose(params['n_filters']*2, kernel_size=2, strides=2, padding='same')(d2)
d3c = Concatenate()([d3CT, a2])
x3 = Dropout(params['dropout'])(d3c)

d3 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x3)

d3 = Conv2D(
    params['n_filters'] * 2, 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d3)

# Decoder4
d4CT = Conv2DTranspose(params['n_filters'], kernel_size=2, strides=2, padding='same')(d3)
d4c = Concatenate()([d4CT, a1])
x4 = Dropout(params['dropout'])(d4c)

d4 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(x4)

d4 = Conv2D(
    params['n_filters'], 
    params['FL'], 
    activation='relu', 
    kernel_initializer=params['init'], 
    kernel_regularizer=l2(params['lmbda']), 
    padding='same'
)(d4)

# Output layer
output = Conv2D(1, 1, activation='sigmoid')(d4)
model = keras.Model(img_input, output)
model.summary()

In [ ]:
# loss is constructed from params so the two can't drift apart
# MLflow logs the params
loss_fn = keras.losses.BinaryFocalCrossentropy(
    apply_class_balancing=params['focal_class_balancing'],
    alpha=params['focal_alpha'],
    gamma=params['focal_gamma'],
)

model.compile(optimizer=keras.optimizers.Adam(params['learning_rate']), loss=loss_fn)

# EarlyStopping: stop once val_loss stops improving for `patience` epochs and roll
# back to the best weights - without restore_best_weights you keep the overfit ones.
# ModelCheckpoint: a multi-day run that crashes at epoch 8 otherwise loses everything.
os.makedirs('checkpoints', exist_ok=True)

callbacks = [
    keras.callbacks.EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(f"checkpoints/{params['model']}_{params['channels']}_{{epoch:02d}}.keras", monitor='val_loss', save_best_only=True, verbose=1),
]


In [ ]:
# SMOKE TEST - not a real run. 8 patches, no augmentation, no callbacks.
# Loss must fall toward 0: 8 patches seen 500 times is trivially memorisable.
# If it plateaus, the bug is upstream (mask alignment, channel order), not the model.
model.fit(
    patchGenerator(train_idx[:8], channels=params['channels'],
                   batch_size=params['batch_size'], augment_data=False),
    steps_per_epoch=10,
    epochs=params['epochs'],
)


## Training

In [ ]:
# MLflow tracks every training run — hyperparameters, metrics, and the model itself
# run `mlflow ui` in the terminal and localhost:5000
# each run is logged separately to compare experiments side by side

# [source]: https://mlflow.org/docs/latest/python_api/mlflow.keras.html
# [example source]: https://github.com/mlflow/mlflow/blob/master/examples/keras/train.py

mlflow.set_experiment('lunar-crater-detection')

with mlflow.start_run(run_name='fusion-wac-dem'):
    mlflow.log_params(params)

    history = model.fit(
        patchGenerator(train_idx, channels=params['channels'],batch_size=params['batch_size']),
        steps_per_epoch=stepsPerEpoch(train_idx, params['batch_size']),
        validation_data=patchGenerator(val_idx, channels=params['channels'],batch_size=params['batch_size'], augment_data=False),
        validation_steps=stepsPerEpoch(val_idx, params['batch_size']),
        epochs=params['epochs'],
        callbacks=callbacks,
    )

    for epoch, (tl, vl) in enumerate(zip(history.history['loss'], history.history['val_loss'])):
        mlflow.log_metric('train_loss', tl, step=epoch)
        mlflow.log_metric('val_loss', vl, step=epoch)

    mlflow.keras.log_model(model, 'model')

## Evaluation